# Malayalam Voice Note Summarizer

Pipeline:
1. Load a Malayalam voice note from `sample_data/`
2. Transcribe it with the Sarvam Speech to Text API (`saaras:v3`)
3. Summarize the transcript into bullet points with the Chat Completions API (`sarvam-105b`)
4. (Optional) Narrate the summary back as audio with the Text to Speech API (`bulbul:v3`)

Outputs are saved to `outputs/`.

In [ ]:
%pip install -r requirements.txt

## Setup

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

from dotenv import load_dotenv
from sarvamai import SarvamAI
from sarvamai.play import save

load_dotenv()

SARVAM_API_KEY = os.getenv("SARVAM_API_KEY")
if not SARVAM_API_KEY:
    raise RuntimeError(
        "Set SARVAM_API_KEY in your environment or .env file before running."
    )

client = SarvamAI(api_subscription_key=SARVAM_API_KEY)

SAMPLE_DIR = Path("sample_data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

LANGUAGE_CODE = "ml-IN"

## 1. Load the voice note

Place a Malayalam `.wav` or `.mp3` file in `sample_data/` and set its
filename below.

In [ ]:
AUDIO_FILENAME = "voice_note.wav"  # change to your file's name
audio_path = SAMPLE_DIR / AUDIO_FILENAME

if not audio_path.exists():
    raise FileNotFoundError(
        f"Put a Malayalam voice note at {audio_path} before running this cell."
    )

print(f"Using audio file: {audio_path}")

## 2. Transcribe (Speech to Text)

The STT API accepts up to ~30 seconds per request, so longer notes are
split into chunks with `ffmpeg` and transcribed one at a time.

In [ ]:
import subprocess


def split_audio_ffmpeg(
    audio_path: Path, chunk_duration: int = 29, output_dir: str = "chunks"
) -> list[Path]:
    out_dir = Path(output_dir)
    out_dir.mkdir(exist_ok=True)
    ext = audio_path.suffix.lower()
    codec = "pcm_s16le" if ext == ".wav" else "libmp3lame"
    output_pattern = out_dir / f"{audio_path.stem}_%03d{ext}"

    command = [
        "ffmpeg", "-y", "-i", str(audio_path),
        "-f", "segment", "-segment_time", str(chunk_duration),
        "-c:a", codec, str(output_pattern),
    ]
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg failed:\n{result.stderr}")

    return sorted(out_dir.glob(f"{audio_path.stem}_*{ext}"))


def transcribe_chunks(chunk_paths: list[Path], language_code: str) -> str:
    parts = []
    for idx, chunk_path in enumerate(chunk_paths):
        print(f"Transcribing chunk {idx + 1}/{len(chunk_paths)}: {chunk_path.name}")
        with open(chunk_path, "rb") as audio_file:
            response = client.speech_to_text.transcribe(
                file=audio_file, model="saaras:v3", language_code=language_code
            )
        parts.append(response.transcript.strip())
    return " ".join(p for p in parts if p)

In [ ]:
chunks = split_audio_ffmpeg(audio_path)
transcript = transcribe_chunks(chunks, LANGUAGE_CODE)

(OUTPUT_DIR / "transcript.txt").write_text(transcript, encoding="utf-8")
print("Transcript:\n")
print(transcript)

## 3. Summarize (Chat Completions)

In [ ]:
SUMMARY_PROMPT = (
    "Summarize the following Malayalam voice note transcript into 3-5 "
    "short bullet points. Reply only in Malayalam.\n\n"
    f"Transcript:\n{transcript}"
)

response = client.chat.completions(
    model="sarvam-105b",
    messages=[
        {"role": "system", "content": "You are a helpful assistant that writes concise summaries in Malayalam."},
        {"role": "user", "content": SUMMARY_PROMPT},
    ],
    temperature=0.3,
)

summary = response.choices[0].message.content.strip()
(OUTPUT_DIR / "summary.txt").write_text(summary, encoding="utf-8")
print("Summary:\n")
print(summary)

## 4. Narrate the summary (Text to Speech, optional)

In [ ]:
tts_response = client.text_to_speech.convert(
    text=summary,
    target_language_code=LANGUAGE_CODE,
    model="bulbul:v3",
    speaker="anushka",
    pace=1.0,
)

save(tts_response, str(OUTPUT_DIR / "summary_audio.wav"))
print(f"Saved narrated summary to {OUTPUT_DIR / 'summary_audio.wav'}")